In [1]:
"""
Clean prices

Transform StatCan's raw retail price file into a
smaller table with 4 columns: product, place, month, price.

"""
import pandas as pd

pd.set_option("display.max_rows", 120)

raw_prices = pd.read_csv("../data/raw/retail_prices/18100245.csv", encoding="utf-8-sig", low_memory=False)
print(raw_prices.shape)

(140124, 15)


In [2]:
# Keep the 4 columns and rename them 
prices = raw_prices[["REF_DATE", "GEO", "Products", "VALUE"]].rename(columns={
    "REF_DATE": "date",
    "GEO": "place",
    "Products": "product",
    "VALUE": "price",
})
prices.head()

,date,place,product,price
0,2017-01,Canada,"Beef stewing cuts, per kilogram",12.66
1,2017-01,Canada,"Beef striploin cuts, per kilogram",21.94
2,2017-01,Canada,"Beef top sirloin cuts, per kilogram",13.44
3,2017-01,Canada,"Beef rib cuts, per kilogram",20.17
4,2017-01,Canada,"Ground beef, per kilogram",9.12


In [3]:
# Convert "2025-03" text into a real date format(first day of the month)
# + trim stray spaces from product names
prices["date"] = pd.to_datetime(prices["date"], format="%Y-%m")
prices["product"] = prices["product"].str.strip()
prices.dtypes

date       datetime64[us]
place                 str
product               str
price             float64
dtype: object

In [4]:
# How many rows have no price?
missing = prices["price"].isna().sum()
print(missing, "rows with no price out of", len(prices))

32 rows with no price out of 140124


In [5]:
# Drop the 32 rows with no price
prices = prices.dropna(subset=["price"])
print(len(prices), "rows left")

140092 rows left


In [6]:
# Identify the first and last month each product has a price (Canada-wide), and how many months before the tariffs
canada = prices[prices["place"] == "Canada"]

coverage = canada.groupby("product")["date"].agg(first_month="min", last_month="max", months="count")
coverage["months_before_tariffs"] = (
    canada[canada["date"] < "2025-03-01"].groupby("product")["date"].count()
)
coverage.sort_values("first_month")

,first_month,last_month,months,months_before_tariffs
product,,,,
"Almonds, 200 grams",2017-01-01,2026-07-01,115,98
"Peppers, per kilogram",2017-01-01,2026-07-01,115,98
"Pears, per kilogram",2017-01-01,2026-07-01,115,98
"Peanuts, 450 grams",2017-01-01,2026-07-01,115,98
"Peanut butter, 1 kilogram",2017-01-01,2026-07-01,115,98
"Pasta sauce, 650 millilitres",2017-01-01,2026-07-01,115,98
"Oranges, per kilogram",2017-01-01,2026-07-01,115,98
"Oranges, 1.36 kilograms",2017-01-01,2026-07-01,115,98
"Orange juice, 2 litres",2017-01-01,2026-07-01,115,98


In [7]:
# Completeness check by region: a full series is 110 products x 115 months = 12,650 rows.
# Regions well below this have gaps and need care in regional comparisons.
prices["place"].value_counts()

place
Canada                                12650
Prince Edward Island                  12650
Nova Scotia                           12650
New Brunswick                         12650
Quebec                                12650
Ontario                               12650
Manitoba                              12650
Saskatchewan                          12650
Alberta                               12650
British Columbia                      12650
Newfoundland and Labrador             12535
Whitehorse, Yukon                       643
Yellowknife, Northwest Territories      414
Name: count, dtype: int64

In [8]:
# Which product is missing from Newfoundland and Labrador? it has 115 fewer rows than other provinces
newfoundland_products = set(prices[prices["place"] == "Newfoundland and Labrador"]["product"])
print(set(prices["product"]) - newfoundland_products)

{'Milk, 4 litres'}


In [9]:
# Drop Whitehorse and Yellowknife: too few published prices for reliable comparisons
prices = prices[~prices["place"].isin(["Whitehorse, Yukon", "Yellowknife, Northwest Territories"])]
print(prices["place"].nunique(), "places left")

11 places left


In [10]:
# Label each price row as tariffed / not tariffed by joining the product-tariff crosswalk (notebook 02).
# This label defines the two groups compared in the analysis. Count of NaN should be 0 (every product matched).
crosswalk = pd.read_csv("../data/reference/product_tariff_crosswalk.csv", dtype=str)
prices = prices.merge(crosswalk[["product", "status"]], on="product", how="left")

print(prices["status"].value_counts(dropna=False))
prices.head()

status
not tariffed    89815
tariffed        49220
Name: count, dtype: int64


,date,place,product,price,status
0,2017-01-01,Canada,"Beef stewing cuts, per kilogram",12.66,not tariffed
1,2017-01-01,Canada,"Beef striploin cuts, per kilogram",21.94,not tariffed
2,2017-01-01,Canada,"Beef top sirloin cuts, per kilogram",13.44,not tariffed
3,2017-01-01,Canada,"Beef rib cuts, per kilogram",20.17,not tariffed
4,2017-01-01,Canada,"Ground beef, per kilogram",9.12,not tariffed


In [11]:
# Save the cleaned, labelled price table as the input for the database load and analysis.
prices.to_csv("../data/processed/prices_clean.csv", index=False)
print("Saved", len(prices), "rows")

Saved 139035 rows
